# Adult Census Income | Random Forests — End-to-End Portfolio Project
**Goal:** Predict whether a person's annual income exceeds **$50,000** using the UCI Adult Census Income data.  
**Scope:** Reproduce the 17 Codecademy steps, then extend the work with leakage-aware preprocessing, feature engineering, cross-validation, and robust evaluation.

**Important:** Numerical outcomes will appear when you **Run All**. This notebook contains no fabricated training results. The public UCI dataset may differ from the version packaged by Codecademy (row count, missing values, or split), so scores need not exactly match earlier screenshots.

**Limitations:** Historical census data is not necessarily representative of current populations. Demographic attributes such as sex and race warrant special care if these predictions are considered for any consequential application. This is an educational modelling exercise, not a recommended income-decision system.

## 0 · Setup and original data download
The first execution requires internet access. Install packages once if needed using the commented `%pip` line. Source: [UCI Adult / Census Income](https://archive.ics.uci.edu/dataset/2/adult). The loader pulls the original `adult.data` and `adult.test` CSV files (using both UCI files), removing the extra terminal period from income labels in the original test file.

In [ ]:
# If needed, uncomment and run once:
# %pip install numpy pandas matplotlib seaborn scikit-learn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, ConfusionMatrixDisplay,
                             RocCurveDisplay)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

SEED = 42
pd.set_option('display.max_columns', 60)
print('Ready. scikit-learn loaded.')

In [ ]:
# Download directly from the UCI repository. Requires network access on first run.
colnames = ['age','workclass','fnlwgt','education','education-num','marital-status',
            'occupation','relationship','race','sex','capital-gain','capital-loss',
            'hours-per-week','native-country','income']
base_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/'
try:
    train_uci = pd.read_csv(base_url + 'adult.data', header=None, names=colnames,
                            skipinitialspace=True, na_values=['?'])
    test_uci = pd.read_csv(base_url + 'adult.test', header=None, names=colnames,
                           skiprows=1, skipinitialspace=True, na_values=['?'])
except Exception as exc:
    raise RuntimeError('UCI download failed. Connect to the internet, then rerun this cell. '
                       'Alternatively download adult.data and adult.test from the UCI source and '
                       'replace the URLs with their local file paths.') from exc

df = pd.concat([train_uci, test_uci], ignore_index=True)
for c in df.select_dtypes(include=['object', 'string']).columns:
    df[c] = df[c].str.strip()
df['income'] = df['income'].str.rstrip('.')
print('Total rows and columns:', df.shape)
print('Income values:', df['income'].value_counts(dropna=False).to_dict())
assert set(df['income'].dropna().unique()) == {'<=50K','>50K'}, 'Unexpected income labels; inspect data!'
assert df['income'].notna().all(), 'Missing target values found.'
display(df.head(3))

## Part A · Reproduce Codecademy Tasks 1–10
We use the same six predictors shown in your earlier feature-importance output (`age`, `capital-gain`, `capital-loss`, `hours-per-week`, `sex`, `race`). The public pooled UCI dataset and split may give different results from Codecademy.

In [ ]:
# Tasks 1–5: distribution, cleaning (already done), features, binary target, split
print('Income distribution (%)')
display((df['income'].value_counts(normalize=True) * 100).round(2).rename('percent'))
feature_cols = ['age','capital-gain','capital-loss','hours-per-week','sex','race']
y = np.where(df['income'].eq('<=50K'), 0, 1)
X = pd.get_dummies(df[feature_cols], drop_first=True, dtype=int)
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=1, stratify=y)
print('Train/test:', x_train.shape, x_test.shape)
print('Positive class proportion train/test:', y_train.mean().round(4), y_test.mean().round(4))

In [ ]:
# Tasks 6–10: baseline, depth loop 1–25, curves, best depth, importances
baseline = RandomForestClassifier(random_state=SEED, n_jobs=-1)
baseline.fit(x_train, y_train)
print('Default forest held-out test accuracy:', round(baseline.score(x_test,y_test),4))

def depth_experiment(xtr, xte, ytr, yte, max_depth=25):
    depths = list(range(1, max_depth + 1))
    accuracy_train, accuracy_test = [], []
    for depth in depths:
        rf = RandomForestClassifier(max_depth=depth, random_state=SEED, n_jobs=-1)
        rf.fit(xtr, ytr)
        accuracy_train.append(accuracy_score(ytr, rf.predict(xtr)))
        accuracy_test.append(accuracy_score(yte, rf.predict(xte)))
    ix = int(np.argmax(accuracy_test))
    print(f'Best (exploratory) depth: {depths[ix]}; test accuracy: {accuracy_test[ix]:.3%}')
    fig, ax = plt.subplots(figsize=(8,4))
    ax.plot(depths,accuracy_test,'bo--',label='Test accuracy')
    ax.plot(depths,accuracy_train,'r*:',label='Train accuracy')
    ax.set(xlabel='max_depth',ylabel='Accuracy',title='Accuracy by tree depth')
    ax.legend(); plt.show()
    rf = RandomForestClassifier(max_depth=depths[ix],random_state=SEED,n_jobs=-1)
    rf.fit(xtr, ytr)
    ranks = pd.DataFrame({'feature':xtr.columns,'importance':rf.feature_importances_})
    display(ranks.sort_values('importance',ascending=False).head(5))
    return {'depth':depths[ix],'accuracy':accuracy_test[ix],'model':rf,'importances':ranks}

original_results = depth_experiment(x_train,x_test,y_train,y_test)

## Part B · Reproduce Tasks 11–17 — education categories
Using the Adult dataset's numeric education scale, create three groups. We keep the same hold-out split and run the full **1–25** depth range prescribed by the exercise (rather than the earlier 1–9 subset). This reproduces the tutorial's *exploratory* method; using the test set to choose depth is **not** a valid final performance estimate. We correct that in Part C.

In [ ]:
df['education_bin'] = pd.cut(df['education-num'], bins=[0,9,13,16],
                             labels=['HS or less','College to Bachelors','Masters or more'])
assert df['education_bin'].notna().all(), 'Some education levels were not binned.'
feature_cols_edu = feature_cols + ['education_bin']
X_edu = pd.get_dummies(df[feature_cols_edu], drop_first=True, dtype=int)
xtr_e, xte_e, ytr_e, yte_e = train_test_split(
    X_edu, y, test_size=0.20, random_state=1, stratify=y)
engineering_results = depth_experiment(xtr_e,xte_e,ytr_e,yte_e)
print(f"Original ~ {original_results['accuracy']:.3%}, with education bin ~ {engineering_results['accuracy']:.3%}")
print('Difference in percentage points:',round(100*(engineering_results['accuracy']-original_results['accuracy']),3))

## Part C · More defensible evaluation (senior-level extension)
### Design rules
1. Make **one stratified 80/20 train/test split** and **never use the test split to select a model**.
2. Compare feature groups using **5-fold cross-validation** exclusively inside the training set.
3. Fit imputation and one-hot encoding **inside each fold** with a `Pipeline` / `ColumnTransformer`.
4. Pick the winning feature group and hyperparameters using **F1 of the >$50K class**; report ROC-AUC too.
5. Evaluate the winning model **once** on the held-out test split.

Note: race/sex are retained in the tutorial groups for comparable modelling, but the potential for disparate errors is discussed below. Do not use these results for decisions about real people.

In [ ]:
# Features are computed from each person's own record, not from target values.
# Age and weekly hours are already numerical; new candidate feature represents yearly
# work intensity, which is a simplistic proxy and NOT an actual observed annual-hours figure.
model_df = df.copy()
model_df['education_bin'] = pd.cut(model_df['education-num'], [0,9,13,16],
    labels=['HS or less','College to Bachelors','Masters or more']).astype('object')
model_df['hours_age_interaction'] = model_df['hours-per-week'] * model_df['age']

feature_sets = {
    'Original six': ['age','capital-gain','capital-loss','hours-per-week','sex','race'],
    'Plus education bin': ['age','capital-gain','capital-loss','hours-per-week','sex','race','education_bin'],
    'Expanded census': ['age','capital-gain','capital-loss','hours-per-week','education-bin-PLACEHOLDER',
                        'workclass','occupation','marital-status','relationship','native-country',
                        'hours_age_interaction']
}
feature_sets['Expanded census'] = [c if c != 'education-bin-PLACEHOLDER' else 'education_bin'
                                   for c in feature_sets['Expanded census']]
all_features = sorted(set(sum(feature_sets.values(), [])))
X_all = model_df[all_features]
y_all = (model_df['income'] == '>50K').astype(int)
X_train, X_test, Y_train, Y_test = train_test_split(
    X_all,y_all,test_size=.20,random_state=SEED,stratify=y_all)
print('Training:',X_train.shape,'Final hold-out:',X_test.shape)
print('Class 1 proportion:',round(Y_train.mean(),4))

In [ ]:
def make_pipeline(cols, **forest_kwargs):
    cat_cols = [c for c in cols if not pd.api.types.is_numeric_dtype(X_train[c])]
    num_cols = [c for c in cols if c not in cat_cols]
    # OneHotEncoder uses a version-compatible sparse flag depending on scikit-learn version.
    import inspect
    sparse_arg = {'sparse_output':False} if 'sparse_output' in inspect.signature(OneHotEncoder).parameters else {'sparse':False}
    preprocessing = ColumnTransformer([
        ('numeric',SimpleImputer(strategy='median'),num_cols),
        ('categorical',Pipeline([
            ('imputer',SimpleImputer(strategy='most_frequent')),
            ('onehot',OneHotEncoder(handle_unknown='ignore',**sparse_arg))
        ]),cat_cols)
    ])
    forest = RandomForestClassifier(random_state=SEED,n_jobs=-1,**forest_kwargs)
    return Pipeline([('preprocess',preprocessing),('forest',forest)])

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
scoring = {'accuracy':'accuracy','f1':'f1','recall':'recall','precision':'precision','roc_auc':'roc_auc'}
rows=[]
for name,cols in feature_sets.items():
    print('Cross-validating:',name)
    result = cross_validate(make_pipeline(cols,n_estimators=150,max_depth=10),
                            X_train[cols],Y_train,cv=cv,scoring=scoring,n_jobs=1)
    row={'features':name}
    for metric in scoring:row[metric]=result['test_'+metric].mean()
    rows.append(row)
cv_table = pd.DataFrame(rows).sort_values('f1',ascending=False)
display(cv_table.style.format({m:'{:.3f}' for m in scoring}))
winning_name=cv_table.iloc[0]['features']
winning_cols=feature_sets[winning_name]
print('Selected feature group by cross-validated F1:',winning_name)

### Hyperparameter search
Randomised search samples **24 combinations**. It optimises cross-validated F1 for the positive class (income >$50K), not test accuracy. This may take a few minutes depending on hardware.

In [ ]:
from scipy.stats import randint
search = RandomizedSearchCV(
    estimator=make_pipeline(winning_cols),
    param_distributions={
        'forest__n_estimators':[100,200,300],
        'forest__max_depth':[6,9,12,16,20,None],
        'forest__max_features':['sqrt',0.5,1.0],
        'forest__min_samples_leaf':[1,2,4,8],
        'forest__min_samples_split':[2,5,10],
        'forest__class_weight':[None,'balanced','balanced_subsample']
    },
    n_iter=24,scoring='f1',cv=cv,random_state=SEED,n_jobs=1,
    verbose=1,refit=True,return_train_score=True
)
search.fit(X_train[winning_cols],Y_train)
print('Best CV F1:',round(search.best_score_,4))
print('Best hyperparameters:')
for key,val in search.best_params_.items():print(' ',key,'=',val)
best_model = search.best_estimator_

### Final hold-out evaluation — execute **once** after choosing the model
These values are the actual independent comparison you should share in a portfolio. Precision/recall are measured for the **>$50K** class. No threshold is tuned against the test data.

In [ ]:
# FINAL EVALUATION. Do not use the following scores to restart hyperparameter tuning.
final_pred = best_model.predict(X_test[winning_cols])
final_probability = best_model.predict_proba(X_test[winning_cols])[:,1]
final_metrics = {
    'accuracy':accuracy_score(Y_test,final_pred),
    'precision_positive':precision_score(Y_test,final_pred,zero_division=0),
    'recall_positive':recall_score(Y_test,final_pred,zero_division=0),
    'f1_positive':f1_score(Y_test,final_pred,zero_division=0),
    'roc_auc':roc_auc_score(Y_test,final_probability)
}
print(pd.Series(final_metrics).round(4))
print('\nClassification report:\n',classification_report(Y_test,final_pred,
    target_names=['Income <=$50K','Income >$50K'],zero_division=0))
ConfusionMatrixDisplay.from_predictions(Y_test,final_pred,display_labels=['<=$50K','>$50K'],cmap='Blues')
plt.title('Final held-out confusion matrix');plt.show()
RocCurveDisplay.from_predictions(Y_test,final_probability)
plt.title('Final held-out ROC curve');plt.show()

### Model interpretability — feature importance
Random Forest impurity-based importances can favour high-cardinality or continuous variables. Treat them as descriptive, not causal. For more defensible interpretation, consider held-out permutation importance or SHAP analysis after model selection.

In [ ]:
names=best_model.named_steps['preprocess'].get_feature_names_out()
importances=best_model.named_steps['forest'].feature_importances_
importance_df=(pd.DataFrame({'feature':names,'importance':importances})
               .sort_values('importance',ascending=False))
display(importance_df.head(15))
importance_df.head(12).sort_values('importance').plot.barh(
    x='feature',y='importance',figsize=(9,5),legend=False)
plt.title('Top 12 Random Forest feature importances');plt.tight_layout();plt.show()

## Responsible modelling and portfolio conclusions
**Potential bias:** The dataset encodes historical income inequalities. Consider inspecting false-positive and false-negative rates separately by sex and race **for auditing**, not using demographic attributes as a basis for consequential decisions. Removing them from the predictor set alone does not guarantee fairness because other variables can act as proxies.

**Practical next steps:**
- Compare Random Forest with dummy-majority, Logistic Regression and boosted tree baselines.
- Audit subgroup recall and false-positive rates with confidence intervals.
- Report uncertainty using repeated stratified cross-validation or bootstrapping (performed on appropriate data partitions).
- Test whether results remain stable when using the **original UCI predefined train/test partition** rather than a random pooled split.
- Document missingness and the limitations of `capital-gain`, `capital-loss`, and `education_bin` for contemporary data.

### Portfolio results template (fill with actual notebook output)
> Using the UCI Adult Census Income dataset, I built and tuned a Random Forest binary classifier to predict income above $50,000. I compared three feature sets using five-fold stratified cross-validation, selected features and hyperparameters by F1, and evaluated the chosen model once on an untouched 20% test set. Final test results: **accuracy = [ ], F1 = [ ], recall = [ ], ROC-AUC = [ ]**. I examined feature importance and documented model limitations, including data age, class imbalance, and potential demographic bias.

**Source:** Becker, B. & Kohavi, R., *Adult* dataset, UCI Machine Learning Repository, https://doi.org/10.24432/C5XW20.